# Pre-processing of Synthea for LLOS Prediction

Adapts the MIMIC IV preprocessing pipeline for Synthea data using SNOMED codes directly.
Target: Predict Long Length of Stay (LLOS = LOS ≥ mean + 2σ)

In [66]:
# Paths configuration
synthea_path = "../output_synthea/csv"
notes_path = "../output_synthea/notes"
output_path = "./datasets/synthea_processed"

import os
os.makedirs(output_path, exist_ok=True)

## 1. Physical characteristics extraction and LOS calculation
Filter to inpatient encounters only, calculate LOS, create LLOS binary label

In [67]:
import pandas as pd
import numpy as np

# Load Synthea data
encounters_df = pd.read_csv(f"{synthea_path}/encounters.csv")
patients_df = pd.read_csv(f"{synthea_path}/patients.csv")

print("Encounters shape:", encounters_df.shape)
print("Patients shape:", patients_df.shape)
print("\nEncounter classes:", encounters_df['ENCOUNTERCLASS'].value_counts())

Encounters shape: (67956, 15)
Patients shape: (1155, 28)

Encounter classes: ENCOUNTERCLASS
ambulatory    36878
wellness      14034
outpatient     8888
urgentcare     3137
emergency      3005
inpatient      1271
home            301
snf             174
hospice         157
virtual         111
Name: count, dtype: int64


In [68]:
# Filter to inpatient encounters only (these have meaningful multi-day LOS)
df_enc = encounters_df[encounters_df['ENCOUNTERCLASS'] == 'inpatient'].copy()
print(f"Inpatient encounters: {len(df_enc)}")

# Rename columns to match MIMIC format
df_enc = df_enc.rename(columns={
    'Id': 'hadm_id',
    'PATIENT': 'subject_id',
    'ENCOUNTERCLASS': 'admission_type',
    'START': 'admittime',
    'STOP': 'dischtime',
    'REASONCODE': 'reason_code',
    'REASONDESCRIPTION': 'reason_description'
})

# Merge with patient demographics
patients_df = patients_df.rename(columns={
    'Id': 'subject_id',
    'GENDER': 'gender',
    'RACE': 'race',
    'ETHNICITY': 'ethnicity',
    'MARITAL': 'marital_status',
    'BIRTHDATE': 'birthdate'
})

df_phy_ad = pd.merge(
    df_enc[['subject_id', 'hadm_id', 'admission_type', 'admittime', 'dischtime', 'reason_code', 'reason_description']],
    patients_df[['subject_id', 'gender', 'race', 'ethnicity', 'marital_status', 'birthdate']],
    on='subject_id',
    how='left'
)

print(f"Merged shape: {df_phy_ad.shape}")

Inpatient encounters: 1271
Merged shape: (1271, 12)


In [69]:
# Calculate LOS in days
df_phy_ad['admittime'] = pd.to_datetime(df_phy_ad['admittime'], utc=True)
df_phy_ad['dischtime'] = pd.to_datetime(df_phy_ad['dischtime'], utc=True)
df_phy_ad = df_phy_ad.dropna(subset=['admittime', 'dischtime']).copy()

df_phy_ad['LOS'] = (df_phy_ad['dischtime'] - df_phy_ad['admittime']).dt.total_seconds() / (3600 * 24)
df_phy_ad['LOS'] = df_phy_ad['LOS'].round(6)

# Calculate age at admission (handle timezone-aware/naive mismatch)
df_phy_ad['birthdate'] = pd.to_datetime(df_phy_ad['birthdate'], utc=True)
df_phy_ad['anchor_age'] = ((df_phy_ad['admittime'] - df_phy_ad['birthdate']).dt.days / 365.25).astype(int)

# Filter reasonable LOS (0-500 days like MIMIC pipeline)
condition = (df_phy_ad['LOS'] >= 0) & (df_phy_ad['LOS'] <= 500)
df_phy_ad = df_phy_ad[condition]

print(f"After LOS filter: {len(df_phy_ad)} encounters")
print(f"\nLOS statistics:")
print(df_phy_ad['LOS'].describe())

After LOS filter: 1271 encounters

LOS statistics:
count    1271.000000
mean        4.613033
std         4.421922
min         1.000000
25%         1.000000
50%         3.380556
75%         6.445486
max        34.000000
Name: LOS, dtype: float64


In [70]:
# Numeric mapping for categorical variables (Synthea-specific values)
gender_nums = {'F': 1, 'M': 2}
race_nums = {'white': 1, 'black': 2, 'asian': 3, 'native': 4, 'other': 5}
marital_nums = {'S': 1, 'M': 2, 'D': 3, 'W': 4}  # Single, Married, Divorced, Widowed
admission_nums = {'inpatient': 1, 'emergency': 2, 'urgentcare': 3, 'ambulatory': 4, 'wellness': 5, 'outpatient': 6}

# Apply mappings
df_phy_ad['gender'] = df_phy_ad['gender'].map(gender_nums).fillna(0).astype(int)
df_phy_ad['race'] = df_phy_ad['race'].str.lower().map(race_nums).fillna(0).astype(int)
df_phy_ad['marital_status'] = df_phy_ad['marital_status'].map(marital_nums).fillna(0).astype(int)
df_phy_ad['admission_type'] = df_phy_ad['admission_type'].str.lower().map(admission_nums).fillna(0).astype(int)

print("Gender distribution:", df_phy_ad['gender'].value_counts().to_dict())
print("Race distribution:", df_phy_ad['race'].value_counts().to_dict())

Gender distribution: {2: 763, 1: 508}
Race distribution: {1: 977, 2: 139, 3: 84, 0: 50, 5: 15, 4: 6}


In [71]:
# Calculate LLOS threshold (mean + 2σ) and create binary label
los_mean = df_phy_ad['LOS'].mean()
los_std = df_phy_ad['LOS'].std()
los_threshold = los_mean + 2 * los_std

print(f"LOS mean: {los_mean:.2f} days")
print(f"LOS std: {los_std:.2f} days")
print(f"LLOS threshold (mean + 2σ): {los_threshold:.2f} days")

# Create binary LLOS label
df_phy_ad['is_llos'] = (df_phy_ad['LOS'] >= los_threshold).astype(int)

print(f"\nLLOS distribution:")
print(df_phy_ad['is_llos'].value_counts())
print(f"LLOS rate: {df_phy_ad['is_llos'].mean()*100:.2f}%")

LOS mean: 4.61 days
LOS std: 4.42 days
LLOS threshold (mean + 2σ): 13.46 days

LLOS distribution:
is_llos
0    1216
1      55
Name: count, dtype: int64
LLOS rate: 4.33%


In [72]:
# Save physical + admission data
cols_to_save = ['subject_id', 'hadm_id', 'admission_type', 'marital_status', 'race', 
                'gender', 'anchor_age', 'admittime', 'dischtime', 'LOS', 'is_llos']
df_phy_ad[cols_to_save].to_csv(f"{output_path}/phy_ad.csv", index=False)

print(f"Saved {len(df_phy_ad)} encounters to phy_ad.csv")
print(f"Unique patients: {df_phy_ad['subject_id'].nunique()}")

Saved 1271 encounters to phy_ad.csv
Unique patients: 432


## 2. Diagnosis SNOMED codes one-hot encoding

In [73]:
import pandas as pd

df_phy = pd.read_csv(f"{output_path}/phy_ad.csv")
conditions_df = pd.read_csv(f"{synthea_path}/conditions.csv")

print("Conditions shape:", conditions_df.shape)
print(conditions_df.head())

Conditions shape: (42962, 7)
        START        STOP                               PATIENT  \
0  1970-09-11  1970-10-16  ba0aca58-4f7d-1eb4-3e2c-7bd93298e87b   
1  1970-12-18  1971-08-20  ba0aca58-4f7d-1eb4-3e2c-7bd93298e87b   
2  1971-06-11         NaN  ba0aca58-4f7d-1eb4-3e2c-7bd93298e87b   
3  1971-06-11         NaN  ba0aca58-4f7d-1eb4-3e2c-7bd93298e87b   
4  1971-11-19         NaN  ba0aca58-4f7d-1eb4-3e2c-7bd93298e87b   

                              ENCOUNTER     SYSTEM        CODE  \
0  ba0aca58-4f7d-1eb4-480b-9870d090781a  SNOMED-CT   314529007   
1  ba0aca58-4f7d-1eb4-a2c3-41c91507d34a  SNOMED-CT   314529007   
2  ba0aca58-4f7d-1eb4-125b-40c1823e23ee  SNOMED-CT   128613002   
3  ba0aca58-4f7d-1eb4-125b-40c1823e23ee  SNOMED-CT  1290882004   
4  ba0aca58-4f7d-1eb4-62c6-5aa512a91164  SNOMED-CT   314529007   

                         DESCRIPTION  
0  Medication review due (situation)  
1  Medication review due (situation)  
2        Seizure disorder (disorder)  
3     History o

In [74]:
# Rename columns to match our format
df_cond = conditions_df.rename(columns={
    'PATIENT': 'subject_id',
    'ENCOUNTER': 'hadm_id',
    'CODE': 'snomed_code',
    'DESCRIPTION': 'description'
})

# Keep only conditions from our filtered encounters
keep_keys = df_phy[['subject_id', 'hadm_id']].drop_duplicates()
df_cond = df_cond.merge(keep_keys, on=['subject_id', 'hadm_id'], how='inner')

# Clean SNOMED codes
df_cond['snomed_code'] = df_cond['snomed_code'].astype(str).str.strip()
df_cond = df_cond.drop_duplicates(subset=['subject_id', 'hadm_id', 'snomed_code'])

print(f"Filtered conditions: {len(df_cond)}")
print(f"Unique SNOMED codes: {df_cond['snomed_code'].nunique()}")

Filtered conditions: 501
Unique SNOMED codes: 31


In [75]:
# One-hot encode SNOMED diagnosis codes
dummies = pd.get_dummies(df_cond['snomed_code'], prefix='d', dtype='uint8')

cond_sparse = pd.concat([df_cond[['subject_id', 'hadm_id']].reset_index(drop=True), dummies], axis=1)
df_cond_encoded = cond_sparse.groupby(['subject_id', 'hadm_id'], sort=False).max().reset_index()

print(f"One-hot encoded diagnoses shape: {df_cond_encoded.shape}")

One-hot encoded diagnoses shape: (420, 33)


In [76]:
# Merge with physical characteristics
merged_df = pd.merge(df_phy, df_cond_encoded, on=['subject_id', 'hadm_id'], how='left')

d_cols = [c for c in merged_df.columns if c.startswith('d_')]
if d_cols:
    merged_df[d_cols] = merged_df[d_cols].fillna(0).astype('uint8')

print(f"Final merged shape: {merged_df.shape}")
merged_df.to_csv(f"{output_path}/phyad_dicd.csv", index=False)
print("Saved phyad_dicd.csv")

Final merged shape: (1271, 42)
Saved phyad_dicd.csv


## 3. Procedure SNOMED codes one-hot encoding

In [77]:
import pandas as pd

df_phyad_dicd = pd.read_csv(f"{output_path}/phyad_dicd.csv")
procedures_df = pd.read_csv(f"{synthea_path}/procedures.csv")

print("Loaded diagnosis-merged dataset:", df_phyad_dicd.shape)
print("Procedures shape:", procedures_df.shape)

Loaded diagnosis-merged dataset: (1271, 42)
Procedures shape: (190143, 10)


In [78]:
# Rename and filter procedures
df_proc = procedures_df.rename(columns={
    'PATIENT': 'subject_id',
    'ENCOUNTER': 'hadm_id',
    'CODE': 'snomed_code',
    'DESCRIPTION': 'description'
})

keep_keys_proc = df_phyad_dicd[['subject_id', 'hadm_id']].drop_duplicates()
df_proc = df_proc.merge(keep_keys_proc, on=['subject_id', 'hadm_id'], how='inner')

df_proc['snomed_code'] = df_proc['snomed_code'].astype(str).str.strip()
df_proc = df_proc.drop_duplicates(subset=['subject_id', 'hadm_id', 'snomed_code'])

print(f"Filtered procedures: {len(df_proc)}")
print(f"Unique procedure codes: {df_proc['snomed_code'].nunique()}")

Filtered procedures: 2957
Unique procedure codes: 99


In [79]:
# One-hot encode procedure codes
p_dummies = pd.get_dummies(df_proc['snomed_code'], prefix='p', dtype='uint8')

proc_block = pd.concat([df_proc[['subject_id', 'hadm_id']].reset_index(drop=True), p_dummies], axis=1)
df_proc_encoded = proc_block.groupby(['subject_id', 'hadm_id'], sort=False).max().reset_index()

print(f"Procedures one-hot encoded shape: {df_proc_encoded.shape}")

Procedures one-hot encoded shape: (859, 101)


In [80]:
# Merge with existing data
merged_df = df_phyad_dicd.merge(df_proc_encoded, on=['subject_id', 'hadm_id'], how='left')

p_cols = [c for c in merged_df.columns if c.startswith('p_')]
if p_cols:
    merged_df[p_cols] = merged_df[p_cols].fillna(0).astype('uint8')

print(f"Final merged shape: {merged_df.shape}")
merged_df.to_csv(f"{output_path}/phyad_dicd_picd.csv", index=False)
print("Saved phyad_dicd_picd.csv")

Final merged shape: (1271, 141)
Saved phyad_dicd_picd.csv


## 4. Medication frequency processing (RxNorm codes)

In [81]:
import pandas as pd
import re

df_other = pd.read_csv(f"{output_path}/phyad_dicd_picd.csv")
medications_df = pd.read_csv(f"{synthea_path}/medications.csv")

print("Medications shape:", medications_df.shape)
print(medications_df.columns.tolist())

Medications shape: (57882, 13)
['START', 'STOP', 'PATIENT', 'PAYER', 'ENCOUNTER', 'CODE', 'DESCRIPTION', 'BASE_COST', 'PAYER_COVERAGE', 'DISPENSES', 'TOTALCOST', 'REASONCODE', 'REASONDESCRIPTION']


In [82]:
# Rename and filter medications
df_med = medications_df.rename(columns={
    'PATIENT': 'subject_id',
    'ENCOUNTER': 'hadm_id',
    'CODE': 'rxnorm_code',
    'DESCRIPTION': 'drug',
    'DISPENSES': 'dispenses',
    'TOTALCOST': 'total_cost',
    'START': 'start_time',
    'STOP': 'stop_time'
})

# Keep only medications from our filtered encounters
valid_pairs = df_other[['subject_id', 'hadm_id']].drop_duplicates()
df_med = df_med.merge(valid_pairs, on=['subject_id', 'hadm_id'], how='inner')

print(f"Filtered medications: {len(df_med)}")

Filtered medications: 7210


In [83]:
# Calculate medication frequency from dispenses and duration
df_med['start_time'] = pd.to_datetime(df_med['start_time'])
df_med['stop_time'] = pd.to_datetime(df_med['stop_time'])

# Duration in days
df_med['duration_days'] = (df_med['stop_time'] - df_med['start_time']).dt.total_seconds() / (3600 * 24)
df_med['duration_days'] = df_med['duration_days'].clip(lower=1)  # Minimum 1 day

# Approximate daily frequency
df_med['dispenses'] = df_med['dispenses'].fillna(1)
df_med['freq_per_day'] = (df_med['dispenses'] / df_med['duration_days']).clip(upper=24)

# Map to frequency codes (similar to MIMIC pipeline)
def freq_to_code(freq):
    try:
        freq = float(freq)
        if freq <= 0: return 0
        elif freq <= 1: return 1  # OD
        elif freq <= 2: return 2  # BID
        elif freq <= 3: return 3  # TID
        elif freq <= 4: return 4  # QID
        elif freq <= 6: return 6  # Q4H
        elif freq <= 8: return 8  # Q3H
        elif freq <= 12: return 12  # Q2H
        else: return 24  # Q1H
    except:
        return 0

df_med['freq_code'] = df_med['freq_per_day'].apply(freq_to_code)

In [84]:
# Create medication identifier (RxNorm code)
df_med['med_id'] = 'rx_' + df_med['rxnorm_code'].astype(str)

# Build frequency matrix
freq_grouped = df_med.groupby(['subject_id', 'hadm_id', 'med_id'])['freq_code'].max().reset_index()
freq_matrix = freq_grouped.pivot(index=['subject_id', 'hadm_id'], columns='med_id', values='freq_code')\
                          .fillna(0).astype(int).reset_index()

print(f"Medication frequency matrix shape: {freq_matrix.shape}")
freq_matrix.to_csv(f"{output_path}/med_freq_matrix.csv", index=False)

Medication frequency matrix shape: (664, 106)


In [85]:
# Merge with existing data
df_data = pd.read_csv(f"{output_path}/phyad_dicd_picd.csv")
df_med_matrix = pd.read_csv(f"{output_path}/med_freq_matrix.csv")

merged_df = pd.merge(df_data, df_med_matrix, on=['subject_id', 'hadm_id'], how='left')
merged_df = merged_df.fillna(0)

print(f"Final merged shape: {merged_df.shape}")
merged_df.to_csv(f"{output_path}/phyad_dicd_picd_medfreq.csv", index=False)
print("Saved phyad_dicd_picd_medfreq.csv")

Final merged shape: (1271, 245)
Saved phyad_dicd_picd_medfreq.csv


## 5. Chief Complaint extraction from Synthea notes

In [86]:
import pandas as pd
import re
import os
from pathlib import Path

df_other = pd.read_csv(f"{output_path}/phyad_dicd_picd_medfreq.csv")

# Load all Synthea notes
notes_files = list(Path(notes_path).glob("*.txt"))
print(f"Found {len(notes_files)} note files")

Found 1155 note files


In [87]:
def extract_patient_id_from_filename(filename):
    """Extract patient UUID from Synthea note filename.
    Format: FirstName_LastName_UUID.txt
    """
    name = filename.stem
    parts = name.split('_')
    if len(parts) >= 3:
        # UUID is the last part (may contain hyphens)
        return '_'.join(parts[2:]) if len(parts) > 3 else parts[2]
    return None

def extract_chief_complaint_synthea(note_text):
    """Extract Chief Complaint from Synthea markdown-formatted notes.
    Synthea uses: # Chief Complaint\nContent\n\n# Next Section
    """
    if not isinstance(note_text, str) or not note_text.strip():
        return "NA"
    
    # Look for markdown header
    match = re.search(r'^#\s*Chief\s+Complaint\s*\n(.+?)(?=^#|\Z)', 
                      note_text, flags=re.MULTILINE | re.IGNORECASE | re.DOTALL)
    
    if match:
        complaint = match.group(1).strip()
        # Clean up
        complaint = re.sub(r'\s+', ' ', complaint)
        return complaint if complaint else "NA"
    
    return "NA"

# Test on first file
if notes_files:
    sample = notes_files[0].read_text(encoding='utf-8')
    print("Sample note (first 500 chars):")
    print(sample[:500])
    print("\nExtracted CC:", extract_chief_complaint_synthea(sample))

Sample note (first 500 chars):

2025-01-16

# Chief Complaint
No complaints.

# History of Present Illness
Rubin812 is a 22 year-old nonhispanic white male. Patient has a history of risk activity involvement (finding), medication review due (situation), primary dental caries (disorder), viral sinusitis (disorder), gingivitis (disorder), fractured dental filling (finding).

# Social History
 Patient has never smoked.
 Patient identifies as heterosexual.

Patient comes from a high socioeconomic background.
 Patient has complete

Extracted CC: No complaints.


In [88]:
# Extract chief complaints from all notes
cc_records = []

for note_file in notes_files:
    try:
        patient_id = extract_patient_id_from_filename(note_file)
        note_text = note_file.read_text(encoding='utf-8')
        cc = extract_chief_complaint_synthea(note_text)
        
        cc_records.append({
            'subject_id': patient_id,
            'chief_complaint': cc,
            'filename': note_file.name
        })
    except Exception as e:
        print(f"Error processing {note_file.name}: {e}")

cc_df = pd.DataFrame(cc_records)
print(f"Extracted {len(cc_df)} chief complaints")
print(f"NA rate: {(cc_df['chief_complaint'] == 'NA').mean()*100:.1f}%")

Extracted 1155 chief complaints
NA rate: 0.0%


In [89]:
# Match with our encounters (by subject_id)
valid_subjects = df_other['subject_id'].unique()
cc_df_filtered = cc_df[cc_df['subject_id'].isin(valid_subjects)]

print(f"Matched {len(cc_df_filtered)} notes to valid patients")
cc_df_filtered.to_csv(f"{output_path}/chief_complaint_extract.csv", index=False)

Matched 280 notes to valid patients


## 6. Chief Complaint cleaning

In [90]:
import nltk
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import pandas as pd
import re

In [91]:
df_cc = pd.read_csv(f"{output_path}/chief_complaint_extract.csv")
complaints = df_cc['chief_complaint'].astype(str).tolist()

stop_words = set(stopwords.words('english'))

# For Synthea, we don't have a medical dictionary, so we'll keep all non-stopwords
# This is simpler but may include non-medical terms
cleaned_complaints = []

for line in complaints:
    if pd.isna(line) or not str(line).strip() or line.upper() == 'NA':
        cleaned_complaints.append("")
        continue
    
    tokens = word_tokenize(line.lower())
    tokens = [re.sub(r'\W+', '', t) for t in tokens if t.isalnum()]
    tokens_nostop = [t for t in tokens if t not in stop_words and len(t) > 2]
    
    cleaned_complaints.append(" ".join(sorted(set(tokens_nostop))))

df_cc['cleaned_complaint'] = cleaned_complaints
df_cc.to_csv(f"{output_path}/chief_complaint_cleaned.csv", index=False)

print(f"Cleaned {sum(1 for c in cleaned_complaints if c)} complaints")

Cleaned 280 complaints


## 7. History of Present Illness extraction

In [92]:
def extract_hpi_synthea(note_text, max_words=200):
    """Extract History of Present Illness from Synthea markdown notes."""
    if not isinstance(note_text, str) or not note_text.strip():
        return "NA"
    
    # Look for HPI section
    match = re.search(r'^#\s*History\s+of\s+Present\s+Illness\s*\n(.+?)(?=^#|\Z)', 
                      note_text, flags=re.MULTILINE | re.IGNORECASE | re.DOTALL)
    
    if match:
        hpi = match.group(1).strip()
        # Limit to max_words
        words = hpi.split()[:max_words]
        hpi = ' '.join(words)
        hpi = re.sub(r'\s+', ' ', hpi)
        return hpi.lower() if hpi else "NA"
    
    return "NA"

# Extract HPI from all notes
hpi_records = []

for note_file in notes_files:
    try:
        patient_id = extract_patient_id_from_filename(note_file)
        note_text = note_file.read_text(encoding='utf-8')
        hpi = extract_hpi_synthea(note_text)
        
        hpi_records.append({
            'subject_id': patient_id,
            'hpi': hpi
        })
    except Exception as e:
        print(f"Error: {e}")

hpi_df = pd.DataFrame(hpi_records)
hpi_df_filtered = hpi_df[hpi_df['subject_id'].isin(valid_subjects)]

print(f"Extracted {len(hpi_df_filtered)} HPI records")
print(f"NA rate: {(hpi_df_filtered['hpi'] == 'NA').mean()*100:.1f}%")

hpi_df_filtered.to_csv(f"{output_path}/hpi_extract.csv", index=False)

Extracted 280 HPI records
NA rate: 0.0%


In [93]:
# Clean HPI
df_hpi = pd.read_csv(f"{output_path}/hpi_extract.csv")
hpis = df_hpi['hpi'].astype(str).tolist()

cleaned_hpi = []
for line in hpis:
    if pd.isna(line) or not str(line).strip() or line.upper() == 'NA':
        cleaned_hpi.append("")
        continue
    
    tokens = word_tokenize(line.lower())
    tokens = [re.sub(r'\W+', '', t) for t in tokens if t.isalnum()]
    tokens_nostop = [t for t in tokens if t not in stop_words and len(t) > 2]
    
    cleaned_hpi.append(" ".join(sorted(set(tokens_nostop))))

df_hpi['cleaned_hpi'] = cleaned_hpi
df_hpi.to_csv(f"{output_path}/hpi_cleaned.csv", index=False)

print(f"Cleaned {sum(1 for c in cleaned_hpi if c)} HPI records")

Cleaned 280 HPI records


## 8. Word2Vec embeddings for CC and HPI

In [94]:
from gensim.models import Word2Vec
import numpy as np

# Combine CC and HPI for training Word2Vec
df_cc = pd.read_csv(f"{output_path}/chief_complaint_cleaned.csv")
df_hpi = pd.read_csv(f"{output_path}/hpi_cleaned.csv")

# Prepare sentences for Word2Vec
sentences = []

for text in df_cc['cleaned_complaint'].dropna():
    if text.strip():
        sentences.append(text.split())

for text in df_hpi['cleaned_hpi'].dropna():
    if text.strip():
        sentences.append(text.split())

print(f"Training Word2Vec on {len(sentences)} sentences")

Training Word2Vec on 560 sentences


In [95]:
# Train Word2Vec model
embedding_dim = 100

if len(sentences) > 0:
    model = Word2Vec(
        sentences=sentences,
        vector_size=embedding_dim,
        window=5,
        min_count=1,
        workers=4,
        epochs=10
    )
    
    # Save model
    model.wv.save_word2vec_format(f"{output_path}/cc_hpi_embed.txt", binary=False)
    print(f"Trained Word2Vec with vocabulary size: {len(model.wv)}")
else:
    print("Warning: No sentences to train Word2Vec")

Trained Word2Vec with vocabulary size: 566


In [96]:
# Load embeddings and compute averaged vectors
word_vectors = {}

try:
    with open(f"{output_path}/cc_hpi_embed.txt", "r", encoding="utf-8") as f:
        next(f)  # Skip header line
        for line in f:
            parts = line.strip().split()
            if len(parts) == embedding_dim + 1:
                word = parts[0]
                vector = np.array(parts[1:], dtype=float)
                word_vectors[word] = vector
    print(f"Loaded {len(word_vectors)} word vectors")
except FileNotFoundError:
    print("No embeddings file found - will use zero vectors")

Loaded 566 word vectors


In [97]:
def average_vector(text, word_vectors, embedding_dim=100):
    if pd.isna(text) or not str(text).strip():
        return np.zeros(embedding_dim)
    
    tokens = str(text).split()
    vectors = [word_vectors[word] for word in tokens if word in word_vectors]
    
    if vectors:
        return np.mean(vectors, axis=0)
    else:
        return np.zeros(embedding_dim)

# Compute CC embeddings
df_cc['embedding'] = df_cc['cleaned_complaint'].apply(lambda x: average_vector(x, word_vectors, embedding_dim))
cc_embedding_df = pd.DataFrame(df_cc['embedding'].tolist(), index=df_cc.index)
cc_embedding_df.columns = [f'cc_embedding_{i}' for i in range(embedding_dim)]

cc_final = pd.concat([df_cc[['subject_id']], cc_embedding_df], axis=1)
cc_final.to_csv(f"{output_path}/averaged_cc_embeds.csv", index=False)
print(f"Saved CC embeddings: {cc_final.shape}")

Saved CC embeddings: (280, 101)


In [98]:
# Compute HPI embeddings
df_hpi['embedding'] = df_hpi['cleaned_hpi'].apply(lambda x: average_vector(x, word_vectors, embedding_dim))
hpi_embedding_df = pd.DataFrame(df_hpi['embedding'].tolist(), index=df_hpi.index)
hpi_embedding_df.columns = [f'hpi_embedding_{i}' for i in range(embedding_dim)]

hpi_final = pd.concat([df_hpi[['subject_id']], hpi_embedding_df], axis=1)
hpi_final.to_csv(f"{output_path}/averaged_hpi_embeds.csv", index=False)
print(f"Saved HPI embeddings: {hpi_final.shape}")

Saved HPI embeddings: (280, 101)


## 9. Final dataset merge

In [99]:
import pandas as pd

# Load all components
df_main = pd.read_csv(f"{output_path}/phyad_dicd_picd_medfreq.csv")
df_cc_embed = pd.read_csv(f"{output_path}/averaged_cc_embeds.csv")
df_hpi_embed = pd.read_csv(f"{output_path}/averaged_hpi_embeds.csv")

print(f"Main data shape: {df_main.shape}")
print(f"CC embeddings shape: {df_cc_embed.shape}")
print(f"HPI embeddings shape: {df_hpi_embed.shape}")

Main data shape: (1271, 245)
CC embeddings shape: (280, 101)
HPI embeddings shape: (280, 101)


In [100]:
# Merge CC embeddings (by subject_id only since notes are per-patient, not per-encounter)
df_merged = df_main.merge(df_cc_embed, on='subject_id', how='left')

# Merge HPI embeddings
df_merged = df_merged.merge(df_hpi_embed, on='subject_id', how='left')

# Fill NaN embeddings with zeros
embedding_cols = [c for c in df_merged.columns if 'embedding' in c]
df_merged[embedding_cols] = df_merged[embedding_cols].fillna(0.0)

print(f"Final merged shape: {df_merged.shape}")

Final merged shape: (1271, 445)


In [101]:
# Save final dataset
df_merged.to_csv(f"{output_path}/final_dataset.csv", index=False)

# Print summary statistics
print("=" * 50)
print("FINAL DATASET SUMMARY")
print("=" * 50)
print(f"Total encounters: {len(df_merged)}")
print(f"Unique patients: {df_merged['subject_id'].nunique()}")
print(f"Total features: {len(df_merged.columns)}")
print(f"\nFeature breakdown:")
print(f"  - Demographics: 7 (subject_id, hadm_id, admission_type, marital_status, race, gender, anchor_age)")
print(f"  - LOS columns: 3 (admittime, dischtime, LOS)")
print(f"  - Target: 1 (is_llos)")
print(f"  - Diagnosis codes (d_*): {len([c for c in df_merged.columns if c.startswith('d_')])}")
print(f"  - Procedure codes (p_*): {len([c for c in df_merged.columns if c.startswith('p_')])}")
print(f"  - Medication codes (rx_*): {len([c for c in df_merged.columns if c.startswith('rx_')])}")
print(f"  - CC embeddings: {len([c for c in df_merged.columns if c.startswith('cc_embedding')])}")
print(f"  - HPI embeddings: {len([c for c in df_merged.columns if c.startswith('hpi_embedding')])}")
print(f"\nLLOS distribution:")
print(df_merged['is_llos'].value_counts())
print(f"LLOS rate: {df_merged['is_llos'].mean()*100:.2f}%")

FINAL DATASET SUMMARY
Total encounters: 1271
Unique patients: 432
Total features: 445

Feature breakdown:
  - Demographics: 7 (subject_id, hadm_id, admission_type, marital_status, race, gender, anchor_age)
  - LOS columns: 3 (admittime, dischtime, LOS)
  - Target: 1 (is_llos)
  - Diagnosis codes (d_*): 31
  - Procedure codes (p_*): 99
  - Medication codes (rx_*): 104
  - CC embeddings: 100
  - HPI embeddings: 100

LLOS distribution:
is_llos
0    1216
1      55
Name: count, dtype: int64
LLOS rate: 4.33%


## 10. Train/Test split (stratified 80/20)

In [102]:
from sklearn.model_selection import train_test_split
import pandas as pd

df = pd.read_csv(f"{output_path}/final_dataset.csv")

# Identify feature columns (exclude IDs, timestamps, LOS, and target)
exclude_cols = ['subject_id', 'hadm_id', 'admittime', 'dischtime', 'LOS', 'is_llos']
feature_cols = [c for c in df.columns if c not in exclude_cols]

X = df[feature_cols]
y = df['is_llos']

print(f"Features shape: {X.shape}")
print(f"Target distribution: {y.value_counts().to_dict()}")

Features shape: (1271, 439)
Target distribution: {0: 1216, 1: 55}


In [103]:
# Stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y
)

print(f"Training set: {len(X_train)} samples")
print(f"  LLOS=0: {(y_train==0).sum()}, LLOS=1: {(y_train==1).sum()}")
print(f"Test set: {len(X_test)} samples")
print(f"  LLOS=0: {(y_test==0).sum()}, LLOS=1: {(y_test==1).sum()}")

Training set: 1016 samples
  LLOS=0: 972, LLOS=1: 44
Test set: 255 samples
  LLOS=0: 244, LLOS=1: 11


In [104]:
# Save train/test splits
train_df = df.iloc[X_train.index]
test_df = df.iloc[X_test.index]

train_df.to_csv(f"{output_path}/train_set.csv", index=False)
test_df.to_csv(f"{output_path}/test_set.csv", index=False)

print(f"Saved train_set.csv ({len(train_df)} rows)")
print(f"Saved test_set.csv ({len(test_df)} rows)")

Saved train_set.csv (1016 rows)
Saved test_set.csv (255 rows)


In [105]:
# Save feature column names for model training
with open(f"{output_path}/feature_columns.txt", 'w') as f:
    f.write('\n'.join(feature_cols))

print(f"Saved {len(feature_cols)} feature column names")
print("\nPreprocessing complete! Ready for Deep Patient model training.")

Saved 439 feature column names

Preprocessing complete! Ready for Deep Patient model training.
